# Seleção de Meta-Features com Algoritmos Genéticos em Meta-Learning

Pipeline completo de meta-aprendizado para recomendação de algoritmos de classificação
usando datasets reais do OpenML.

### Visão geral do pipeline

```
OpenML (500 datasets)
        │
        ▼
[Fase 1] Avaliação dos base learners → meta_y (melhor algoritmo por dataset)
        │
        ▼
[Fase 2] Extração de meta-features com pymfe → meta_X (matriz de descrição)
        │
        ▼
[Fase 3] Pré-processamento: imputação → variância → correlação → escala
        │
        ▼
[Fase 4] Algoritmo Genético (com caching) seleciona subconjunto de meta-features
        │
        ▼
[Fase 5] StratifiedKFold 5-fold compara GA vs Baseline vs Random vs Feature Importance
```

### Tabela de melhorias implementadas nesta versão

| Componente | Mudança | Motivo |
|---|---|---|
| `OUTER_CV` | `KFold` → `StratifiedKFold` | Meta-dataset desbalanceado; KFold criava folds sem classes minoritárias |
| `META_MODEL` | `max_depth=None, leaf=1` → `max_depth=15, leaf=2, class_weight='balanced'` | Evita overfitting; corrige viés para classes dominantes |
| `INNER_META_CV` | 3 folds → 5 folds | Fitness menos ruidoso; GA otimiza sinal real |
| `GradientBoostingClassifier` | → `HistGradientBoostingClassifier` | 5–10× mais rápido; labels mais precisas |
| `ExtraTrees` | 250 → 100 estimadores | Suficiente para CV de 5 folds; economiza tempo |
| `build_base_pipeline` | `KNNImputer` → `SimpleImputer(median)` | KNNImputer desnecessariamente lento para gerar meta-labels |
| `fit_meta_preprocessor` | + filtro de correlação `|r| > 0.90` | Remove redundâncias antes do GA; reduz espaço de busca de ~148 → ~90 features |
| `build_openml_catalog` | Removida ordenação por tamanho | Evitava viés para datasets minúsculos com meta-features instáveis |
| **GA caching** | `dict` por hash da máscara | Evita recalcular fitness de soluções repetidas (elitismo, convergência) |
| **GA fitness** | `accuracy` → `f1_macro` | F1-macro penaliza erros nas classes minoritárias; cria gradiente mais rico |
| **GA penalidade** | fixa `0.03` → dinâmica escalonada | Penaliza quadraticamente soluções com >40% das features |
| **GA população inicial** | 50% ativas → 35% ativas | Explora soluções compactas desde a geração 0 |
| `crossover_type` | `single_point` → `two_points` | Preserva blocos de features cooperantes |
| `num_generations` | 30 → 50 | Mais orçamento de busca |
| `sol_per_pop` | 20 → 30 | Maior diversidade genética |
| `keep_elitism` | ausente → 2 | 2 melhores indivíduos sempre sobrevivem |
| `stop_criteria` | `saturate_8` → `saturate_15` | Parava cedo demais; aguarda estagnação real |
| Histórico GA | apenas fold 1 → todos os folds | Diagnóstico completo de convergência |

> **Checkpointing:** cada etapa salva artefatos em disco. Execuções seguintes carregam
> do cache, pulando o processamento já concluído.


## 0. Instalação de dependências

In [ ]:
# Execute apenas na primeira vez ou ao atualizar o ambiente.
!pip install -U openml pandas scikit-learn joblib seaborn matplotlib pymfe pygad -q

## 1. Imports e configuração global

In [ ]:
# ── Biblioteca padrão ──────────────────────────────────────────────────────
import os           # variáveis de ambiente e contagem de CPUs
import re           # expressões regulares para normalização de nomes
import json         # serialização legível de objetos simples
import random       # semente global de aleatoriedade
import pickle       # serialização binária de objetos Python complexos
import warnings     # supressão de avisos irrelevantes
from pathlib import Path           # manipulação portável de caminhos
from collections import Counter    # contagem de frequência das meta-features selecionadas
from dataclasses import dataclass  # estrutura de dados sem boilerplate
from typing import Dict, List, Optional, Tuple

# ── Computação numérica e dados ────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualização ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── OpenML, meta-features e GA ─────────────────────────────────────────────
import openml
import pygad
from pymfe.mfe import MFE

# ── Scikit-learn ───────────────────────────────────────────────────────────
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier,  # substitui GradientBoostingClassifier
)
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler,
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Semente única para todas as fontes de aleatoriedade do experimento.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

TARGET_DATASETS   = 500
OPENML_SAMPLE_CAP = 650

# N_JOBS: paralelismo externo por dataset; internamente cada modelo usa n_jobs=1
# para evitar oversubscription (mais threads que CPUs físicos).
N_JOBS = max(1, (os.cpu_count() or 2) - 1)

# ── Diretórios de artefatos ────────────────────────────────────────────────
PROJECT_ROOT            = Path.cwd()
CACHE_DIR               = PROJECT_ROOT / "artifacts_meta_learning"
DATASETS_DIR            = CACHE_DIR / "datasets"
META_DIR                = CACHE_DIR / "meta"
RESULTS_DIR             = CACHE_DIR / "results"
INTERMEDIATE_TABLES_DIR = PROJECT_ROOT / "intermidiate_tables_csv"

for folder in [CACHE_DIR, DATASETS_DIR, META_DIR, RESULTS_DIR, INTERMEDIATE_TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MPL_TMP = os.path.join(os.getcwd(), ".mplconfig")
os.makedirs(MPL_TMP, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", MPL_TMP)

print(f"N_JOBS configurado para: {N_JOBS}")
print(f"Cache local: {CACHE_DIR}")


## 2. Configurações experimentais

In [ ]:
# ── Algoritmos do nível-base ───────────────────────────────────────────────
# Cada algoritmo é avaliado em todos os 500 datasets via cross-validation.
# O algoritmo com maior acurácia média vira a meta-label daquele dataset.
#
# MELHORIAS:
#   • HistGradientBoostingClassifier substitui GradientBoostingClassifier:
#     5–10× mais rápido, suporta NaN nativamente, geralmente mais preciso.
#     Como GradientBoosting domina o meta-dataset (~193 vitórias), usar
#     uma versão mais forte dele gera meta-labels mais precisas.
#   • ExtraTrees reduzido de 250 para 100 estimadores: suficiente para
#     StratifiedKFold de 5 folds, economizando tempo sem perda de qualidade.
BASE_MODELS = {
    "DecisionTree":         DecisionTreeClassifier(random_state=RANDOM_STATE),
    "SVC":                  SVC(gamma="scale", random_state=RANDOM_STATE),
    "KNN":                  KNeighborsClassifier(),
    "LogisticRegression":   LogisticRegression(
                                max_iter=1500, solver="lbfgs",
                                random_state=RANDOM_STATE),
    "ExtraTrees":           ExtraTreesClassifier(
                                n_estimators=100,        # era 250
                                random_state=RANDOM_STATE,
                                n_jobs=1),
    "HistGradientBoosting": HistGradientBoostingClassifier(
                                random_state=RANDOM_STATE),  # era GradientBoostingClassifier
    "GaussianNB":           GaussianNB(),
}

# ── Meta-modelo ────────────────────────────────────────────────────────────
# RandomForest usado tanto no fitness do GA quanto na avaliação final.
#
# MELHORIAS:
#   • max_depth=15 (era None): evita overfitting com ~400 amostras de treino.
#     Árvores sem limite de profundidade memorizam o treino em meta-datasets
#     pequenos, gerando um meta-modelo que não generaliza bem.
#   • min_samples_leaf=2 (era 1): regularização adicional nas folhas.
#   • class_weight='balanced': peso inversamente proporcional à frequência
#     de cada classe. Corrige o viés gerado pelo desbalanceamento do
#     meta-dataset (GradientBoosting+ExtraTrees = ~72% das labels).
#     Sem isso, o meta-modelo tende a ignorar classes minoritárias como
#     KNN (2 ocorrências) e GaussianNB (6 ocorrências).
META_MODEL = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,             # era None
    min_samples_leaf=2,       # era 1
    class_weight="balanced",  # NOVO: compensa desbalanceamento do meta-dataset
    random_state=RANDOM_STATE,
    n_jobs=1,
)

# ── Esquemas de cross-validation ───────────────────────────────────────────
# OUTER_CV: avalia generalização do meta-modelo em datasets nunca vistos.
# MELHORIA: KFold → StratifiedKFold.
# Com 7 classes desbalanceadas no meta-dataset, o KFold sorteava folds
# onde classes minoritárias desapareciam do treino — causa provável da
# queda para 0.57 no fold 5. StratifiedKFold garante proporção de classes
# em todos os folds.
OUTER_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# INNER_META_CV: estima o fitness de cada solução do GA dentro do treino.
# MELHORIA: 3 → 5 folds. Com ~400 amostras e 7 classes, 3 folds gera ~130
# amostras de validação por fold — estimativa muito ruidosa. O GA acabava
# otimizando ruído estatístico em vez de sinal real de qualidade de seleção.
INNER_META_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# BASE_CV: avalia cada base learner em cada dataset do OpenML.
BASE_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── Configuração do Algoritmo Genético ────────────────────────────────────
# Representação: vetor binário de tamanho n_features.
# Gene = 1: meta-feature incluída na seleção.
# Gene = 0: meta-feature excluída.
# O GA maximiza: F1_macro(meta-modelo com features selecionadas) - penalidade
#
# MELHORIAS:
#   • num_generations 30→50: mais orçamento antes de parar.
#   • sol_per_pop 20→30: maior diversidade genética na população inicial.
#   • num_parents_mating 8→10: proporcional ao aumento da população.
#   • keep_elitism=2 (NOVO): os 2 melhores sempre sobrevivem intactos.
#     Evita que boas soluções sejam destruídas por mutação nas gerações tardias.
#   • crossover_type two_points (era single_point): dois pontos de corte
#     preservam melhor blocos de features que cooperam entre si.
#   • mutation_probability[0] 0.20→0.25: mais exploração nas gerações iniciais.
#   • stop_criteria saturate_15 (era saturate_8): aguarda 15 gerações sem
#     melhora antes de encerrar. saturate_8 parava cedo demais, antes de
#     o GA ter chance de escapar de ótimos locais por mutação.
GA_CONFIG = {
    "num_generations":      50,
    "sol_per_pop":          30,
    "num_parents_mating":   10,
    "parent_selection_type": "sss",    # steady-state: mantém diversidade
    "keep_elitism":         2,         # NOVO
    "crossover_type":       "two_points",  # era single_point
    "mutation_type":        "adaptive",
    "mutation_probability": [0.25, 0.05],  # era [0.20, 0.05]
    "gene_space":           [0, 1],
    "suppress_warnings":    True,
    "stop_criteria":        ["saturate_15"],  # era saturate_8
}

# ── Penalidade dinâmica do fitness do GA ──────────────────────────────────
# Penalidade fixa (alpha=0.03) não diferenciava soluções com 20% vs 70%
# das features — o GA não tinha incentivo para compactar além de um certo ponto.
#
# Penalidade dinâmica:
#   • frac ≤ GA_PENALTY_THRESHOLD (40%): penalty = alpha × frac  [linear suave]
#   • frac > GA_PENALTY_THRESHOLD:       penalty = alpha×thresh + alpha×2×(frac-thresh)²
#                                                  [quadrática crescente]
# Isso cria um gradiente claro: abaixo de 40% a penalidade cresce devagar
# (incentiva compactação moderada), acima de 40% cresce rapidamente
# (desincentiva fortemente soluções gordas).
GA_PENALTY_ALPHA     = 0.05   # coeficiente base da penalidade
GA_PENALTY_THRESHOLD = 0.40   # fração de features acima da qual cresce quadraticamente

# Grupos de meta-features do pymfe:
#   general:     estrutura básica do dataset (n_features, n_instances, n_classes…)
#   statistical: momentos e correlações entre atributos numéricos
#   info-theory: entropia, informação mútua, joint entropy
#   landmarking: acurácia de modelos simples como proxy de complexidade
META_FEATURE_GROUPS = ["general", "statistical", "info-theory", "landmarking"]

# Limiar de correlação de Pearson para remover meta-features redundantes.
# Pares com |r| > 0.90 carregam informação quase idêntica; um deles é descartado.
# Isso reduz o espaço de busca do GA de ~148 para ~90-100 features.
CORRELATION_THRESHOLD = 0.90


## 3. Utilitários de cache em disco

Cada etapa pesada persiste seus artefatos como arquivos `.pkl`. Nas execuções
seguintes o notebook detecta os arquivos e pula o processamento já concluído.


In [ ]:
def save_pickle(obj, path: Path) -> None:
    """Persiste qualquer objeto Python em formato binário (pickle)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(obj, f)

def load_pickle(path: Path):
    """Carrega um objeto previamente serializado com save_pickle."""
    with path.open("rb") as f:
        return pickle.load(f)

def save_json(obj, path: Path) -> None:
    """Persiste listas/dicionários simples em JSON legível por humanos."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: Path):
    """Carrega um objeto previamente salvo com save_json."""
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def save_dataframe(df: pd.DataFrame, path: Path) -> None:
    """Persiste um DataFrame preservando índices, dtypes e metadados."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_pickle(path)

def load_dataframe(path: Path) -> pd.DataFrame:
    """Carrega um DataFrame previamente salvo com save_dataframe."""
    return pd.read_pickle(path)

# ── Caminhos dos artefatos (numerados para rastreabilidade) ────────────────
CATALOG_PATH              = CACHE_DIR / "01_openml_catalog_filtered_and_deduplicated.pkl"
DATASETS_PATH             = DATASETS_DIR / "02_real_openml_datasets_downloaded_and_validated.pkl"
BASE_RESULTS_PATH         = META_DIR / "03_base_level_model_evaluation_results.pkl"
BASE_RESULTS_DF_PATH      = META_DIR / "03_base_level_model_evaluation_results_dataframe.pkl"
META_Y_PATH               = META_DIR / "04_meta_labels_best_base_learner_per_dataset.pkl"
META_X_PATH               = META_DIR / "05_meta_features_matrix_extracted_with_pymfe.pkl"
OUTER_RESULTS_PATH        = RESULTS_DIR / "06_outer_cv_ga_feature_selection_evaluation_results.pkl"
TOP_FEATURES_COUNTER_PATH = RESULTS_DIR / "07_ga_selected_meta_feature_frequency_counter.pkl"
ALL_FOLDS_HISTORY_PATH    = RESULTS_DIR / "08_ga_convergence_history_all_folds.pkl"


## 4. Seleção de datasets no OpenML

Busca e filtra datasets de classificação que satisfaçam critérios controlados
de tamanho, dimensão e balanceamento de classes.


In [ ]:
def normalize_name(name: object) -> str:
    """
    Normaliza o nome de um dataset para identificar duplicatas semânticas.

    Remove sufixos de versão ('_v2', '-v3'), substitui separadores por '_'
    e converte para minúsculas. Exemplos: 'Iris_v2' e 'iris-V3' → 'iris'.
    Usado para deduplicar o catálogo mantendo apenas a versão mais recente
    de cada dataset.
    """
    if pd.isna(name):
        return "unknown"
    name = str(name).lower()
    name = re.sub(r"[_\- ]v\d+", "", name)
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


def build_openml_catalog(target_size: int, sample_cap: int, seed: int) -> pd.DataFrame:
    """
    Constrói um catálogo filtrado de datasets candidatos do OpenML.

    Filtros de qualidade aplicados:
      • 200 ≤ instâncias ≤ 10.000  — tamanho computacionalmente viável
      • 5 ≤ features ≤ 100         — complexidade moderada
      • 2 ≤ classes ≤ 10           — classificação multiclasse controlada
      • classe minoritária ≥ 20    — permite StratifiedKFold sem erros
      • exclui datasets FOREX      — séries temporais, incompatíveis com pymfe

    Deduplicação semântica: mantém apenas a versão mais recente de cada
    nome normalizado (evita múltiplas versões do mesmo problema no meta-dataset).

    MELHORIA em relação à versão anterior:
      A versão original ordenava o catálogo por tamanho (NumberOfInstances,
      NumberOfFeatures) antes de baixar. Isso fazia o pipeline processar
      sistematicamente os datasets MENORES primeiro, gerando meta-features
      instáveis (alta variância com poucos dados). Agora embaralhamos
      aleatoriamente, garantindo diversidade de tamanho entre os 500 datasets.
    """
    print("Carregando catálogo do OpenML...")
    df = openml.datasets.list_datasets(output_format="dataframe")

    required_cols = [
        "did", "name", "version",
        "NumberOfInstances", "NumberOfFeatures",
        "NumberOfClasses", "MinorityClassSize",
    ]
    df = df[required_cols].copy()

    df = df[
        df["NumberOfInstances"].between(200, 10000)
        & df["NumberOfFeatures"].between(5, 100)
        & df["NumberOfClasses"].between(2, 10)
        & (df["MinorityClassSize"] >= 20)
        & (~df["name"].str.contains("FOREX", case=False, na=False))
    ].copy()

    df["normalized_name"] = df["name"].apply(normalize_name)
    df = df.drop_duplicates(subset=["did"])
    df = df.sort_values("version", ascending=False).drop_duplicates(
        subset=["normalized_name"]
    )

    if len(df) > sample_cap:
        df = df.sample(n=sample_cap, random_state=seed)

    # Embaralha sem ordenar por tamanho — garante diversidade de datasets.
    return df.sample(frac=1, random_state=seed + 1).reset_index(drop=True)


if CATALOG_PATH.exists():
    catalog_df = load_dataframe(CATALOG_PATH)
    print(f"Catálogo carregado do cache: {CATALOG_PATH}")
else:
    catalog_df = build_openml_catalog(TARGET_DATASETS, OPENML_SAMPLE_CAP, RANDOM_STATE)
    save_dataframe(catalog_df, CATALOG_PATH)
    print(f"Catálogo salvo em: {CATALOG_PATH}")

print(f"Candidatos no catálogo: {len(catalog_df)}")
catalog_df.head()


## 5. Download e validação dos datasets

Cada dataset é baixado individualmente do OpenML, sanitizado e validado.
Datasets que falham em qualquer critério são descartados silenciosamente
para não interromper o pipeline.


In [ ]:
def _coerce_target(y: object) -> pd.Series:
    """
    Normaliza o target para pd.Series unidimensional com índice resetado.

    O OpenML retorna targets como Series, DataFrame ou array dependendo
    do dataset. Esta função unifica todos os casos e rejeita targets
    multi-rótulo (múltiplas colunas), que este experimento não suporta.
    """
    if isinstance(y, pd.Series):
        return y.reset_index(drop=True)
    if isinstance(y, pd.DataFrame):
        if y.shape[1] != 1:
            raise ValueError("Target multirrótulo não suportado.")
        return y.iloc[:, 0].reset_index(drop=True)
    return pd.Series(y).reset_index(drop=True)


def _sanitize_dataframe(X: object) -> pd.DataFrame:
    """
    Padroniza o DataFrame de features com colunas nomeadas como strings.

    Alguns datasets retornam arrays numpy ou DataFrames com índices inteiros.
    Converter para strings garante compatibilidade com ColumnTransformer e pymfe.
    """
    df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
    df.columns = [str(col) for col in df.columns]
    return df.reset_index(drop=True)


def download_openml_dataset(row: pd.Series) -> Optional[Dict]:
    """
    Baixa, valida e padroniza um único dataset do OpenML.

    Validações após o download (reconfirma após limpeza do target):
      • len(X) == len(y): consistência entre features e labels
      • 200 ≤ instâncias ≤ 10.000: tamanho ainda dentro do range
      • 5 ≤ features ≤ 100: dimensão dentro do range
      • 2 ≤ classes únicas ≤ 10: problema de classificação válido
      • classe minoritária ≥ 20: StratifiedKFold possível

    Retorna None em qualquer falha (download, conversão ou validação).
    """
    did = int(row["did"])
    try:
        dataset = openml.datasets.get_dataset(did, download_data=True)
        X, y, _, _ = dataset.get_data(
            dataset_format="dataframe",
            target=dataset.default_target_attribute,
        )
    except Exception as exc:
        print(f"[skip] did={did} | erro no download: {exc}")
        return None

    try:
        X = _sanitize_dataframe(X)
        y = _coerce_target(y)
    except Exception as exc:
        print(f"[skip] did={did} | erro na conversão: {exc}")
        return None

    if len(X) != len(y):
        return None
    if not (200 <= X.shape[0] <= 10000) or not (5 <= X.shape[1] <= 100):
        return None

    # Remove linhas com target inválido e revalida as condições de balanceamento.
    y = y.replace([np.inf, -np.inf], np.nan)
    valid_mask = y.notna()
    X = X.loc[valid_mask].reset_index(drop=True)
    y = y.loc[valid_mask].reset_index(drop=True)

    if y.nunique() < 2 or y.nunique() > 10:
        return None
    if y.value_counts().min() < 20:
        return None

    return {
        "openml_id": did,
        "name": f"openml_{did}_{normalize_name(row['name'])}",
        "data": X,
        "target": y,
    }


if DATASETS_PATH.exists():
    datasets = load_pickle(DATASETS_PATH)
    print(f"Datasets carregados do cache: {DATASETS_PATH}")
else:
    datasets = []
    for _, row in catalog_df.iterrows():
        ds = download_openml_dataset(row)
        if ds is not None:
            datasets.append(ds)
        if len(datasets) >= TARGET_DATASETS:
            break
    save_pickle(datasets, DATASETS_PATH)
    print(f"Datasets salvos em: {DATASETS_PATH}")

print(f"Datasets válidos: {len(datasets)}")
datasets[0]["data"].head() if datasets else None


## 6. Funções auxiliares de pré-processamento

In [ ]:
def infer_column_types(df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    """
    Identifica colunas categóricas e numéricas pelo dtype do pandas.

    Categóricas (precisam de encoding): object, category, bool.
    Numéricas (passam direto): float, int e derivados.
    """
    categorical_cols = df.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()
    numeric_cols = [c for c in df.columns if c not in categorical_cols]
    return categorical_cols, numeric_cols


def encode_target(y: pd.Series) -> np.ndarray:
    """
    Codifica o target em inteiros contíguos [0, n_classes-1].

    Converter para str antes garante compatibilidade quando o OpenML
    retorna tipos mistos (int, float, str) na mesma coluna categórica.
    """
    return LabelEncoder().fit_transform(
        pd.Series(y).astype(str).reset_index(drop=True)
    )


def build_base_pipeline(model) -> Pipeline:
    """
    Constrói o pipeline de pré-processamento para avaliação dos base learners.

    MELHORIA: substituímos KNNImputer por SimpleImputer(median) nos numéricos.
    O KNNImputer é 10–100× mais lento e não altera o ranking dos algoritmos
    (não muda quem é o melhor), apenas adiciona custo sem benefício nesta etapa.

    Numéricos:    imputa pela mediana (robusto a outliers, muito mais rápido)
    Categóricos:  imputa pela moda + one-hot encoding
    Desconhecidas: descartadas (remainder='drop')
    """
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),  # era KNNImputer
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer,
             lambda X: X.select_dtypes(exclude=["object","category","bool"]).columns),
            ("cat", categorical_transformer,
             lambda X: X.select_dtypes(include=["object","category","bool"]).columns),
        ],
        remainder="drop",
    )
    return Pipeline([("preprocessor", preprocessor), ("model", model)])


def prepare_dataset_for_mfe(df: pd.DataFrame) -> np.ndarray:
    """
    Converte o dataset para matriz numérica densa compatível com o pymfe.

    O MFE exige numpy array sem NaN e sem colunas categóricas cruas.

    Processo:
      1. Categóricos: imputa pela moda → OrdinalEncoder
         (preserva estrutura ordinal para métricas de correlação do MFE)
      2. Numéricos: converte para float explicitamente
      3. KNNImputer (n_neighbors=5): preenche NaNs restantes via vizinhos próximos.
         Aqui KNNImputer É justificado — ao contrário do nível-base, a qualidade
         das meta-features depende de uma imputação mais rica: métricas como
         correlação e entropia são sensíveis à distribuição dos dados preenchidos.
    """
    categorical_cols, numeric_cols = infer_column_types(df)
    work_df = df.copy()

    if categorical_cols:
        work_df[categorical_cols] = SimpleImputer(
            strategy="most_frequent"
        ).fit_transform(work_df[categorical_cols])
        work_df[categorical_cols] = OrdinalEncoder(
            handle_unknown="use_encoded_value", unknown_value=-1
        ).fit_transform(work_df[categorical_cols])

    if numeric_cols:
        work_df[numeric_cols] = work_df[numeric_cols].apply(
            pd.to_numeric, errors="coerce"
        )

    return np.asarray(
        KNNImputer(n_neighbors=5).fit_transform(work_df), dtype=float
    )


## 7. Fase 1 — Geração das meta-labels (`meta_y`)

Avalia todos os base learners em cada dataset e registra o mais acurado.
Esse nome (ex: 'HistGradientBoosting') vira a meta-label do dataset.


In [ ]:
def evaluate_base_learners(dataset: Dict) -> Dict:
    """
    Avalia todos os algoritmos candidatos em um dataset e retorna o melhor.

    Para cada algoritmo em BASE_MODELS:
      1. Constrói o pipeline com pré-processamento (build_base_pipeline)
      2. Executa cross_val_score com BASE_CV (StratifiedKFold 5-fold)
      3. Registra a acurácia média como score neste dataset

    O algoritmo com maior acurácia média vira a meta-label do dataset.

    n_jobs=1: o paralelismo externo (Parallel por dataset) já distribui
    a carga entre os CPUs. Usar n_jobs>1 internamente causaria
    oversubscription (mais threads que CPUs físicos), degradando performance.
    """
    X = dataset["data"]
    y = encode_target(dataset["target"])

    scores = {}
    for name, model in BASE_MODELS.items():
        cv_scores = cross_val_score(
            build_base_pipeline(clone(model)), X, y,
            cv=BASE_CV, scoring="accuracy", n_jobs=1,
        )
        scores[name] = float(np.mean(cv_scores))

    best = max(scores, key=scores.get)
    return {"dataset_name": dataset["name"], "best_model": best, "scores": scores}


if (BASE_RESULTS_PATH.exists()
        and BASE_RESULTS_DF_PATH.exists()
        and META_Y_PATH.exists()):
    print("Base-level results encontrados no cache, carregando...")
    base_results    = load_pickle(BASE_RESULTS_PATH)
    base_results_df = load_dataframe(BASE_RESULTS_DF_PATH)
    meta_y          = np.array(load_pickle(META_Y_PATH))
    print(f"Carregados de: {BASE_RESULTS_PATH}")
else:
    print("Avaliando base learners nos datasets (pode demorar)...")
    base_results = Parallel(n_jobs=N_JOBS, verbose=10)(
        delayed(evaluate_base_learners)(ds) for ds in datasets
    )
    meta_y = np.array([r["best_model"] for r in base_results])
    base_results_df = pd.DataFrame([
        {"dataset_name": r["dataset_name"], **r["scores"], "meta_label": r["best_model"]}
        for r in base_results
    ])
    save_pickle(base_results, BASE_RESULTS_PATH)
    save_dataframe(base_results_df, BASE_RESULTS_DF_PATH)
    save_pickle(meta_y.tolist(), META_Y_PATH)
    print(f"Resultados salvos em: {BASE_RESULTS_PATH}")

print("\nDistribuição das meta-labels:")
print(pd.Series(meta_y).value_counts())
base_results_df.head()


## 8. Fase 2 — Extração das meta-features (`meta_X`)

O pymfe calcula métricas descritivas de cada dataset em 4 grupos.
Para métricas que retornam uma distribuição (ex: correlações de todos
os pares de atributos), calculamos 4 resumos: mean, sd, min, max —
multiplicando a riqueza descritiva do meta-dataset.


In [ ]:
def extract_meta_features(dataset: Dict, groups: List[str]) -> Dict:
    """
    Extrai meta-features de um dataset usando o pymfe (MFE).

    Grupos e o que cada um captura:
      general:     estrutura básica — n_features, n_instances, n_classes,
                   razão instâncias/features, proporção de categóricos, etc.
      statistical: momentos estatísticos e correlações entre atributos
                   numéricos (média, variância, curtose, assimetria, etc.)
      info-theory: entropia das classes, informação mútua atributo-classe,
                   joint entropy — descrevem a estrutura de informação do dataset
      landmarking: acurácia de classificadores simples (1-NN, árvore rasa,
                   Naive Bayes) como proxies rápidas de complexidade do problema

    Resumos (mean, sd, min, max): meta-features que retornam uma distribuição
    de valores (ex: correlação de cada par de features) são sumarizadas em 4
    estatísticas, aumentando a representatividade sem perder informação.

    Retorna dicionário com nomes e valores das meta-features extraídas.
    """
    X_num = prepare_dataset_for_mfe(dataset["data"])
    y     = encode_target(dataset["target"])

    mfe = MFE(groups=groups, summary=["mean", "sd", "min", "max"])
    mfe.fit(X_num, y)
    names, values = mfe.extract()

    return {
        "dataset_name":   dataset["name"],
        "feature_names":  list(names),
        "feature_values": list(values),
    }


if META_X_PATH.exists():
    meta_X = load_dataframe(META_X_PATH)
    print(f"meta_X carregado do cache: {META_X_PATH}")
else:
    print("Extraindo meta-features dos datasets (pode demorar)...")
    meta_feature_results = Parallel(n_jobs=N_JOBS, verbose=10)(
        delayed(extract_meta_features)(ds, META_FEATURE_GROUPS) for ds in datasets
    )

    # Verifica que todos os datasets geraram o mesmo conjunto de meta-features.
    # Inconsistência indica versões diferentes do pymfe ou datasets problemáticos.
    reference_names = meta_feature_results[0]["feature_names"]
    for item in meta_feature_results[1:]:
        if item["feature_names"] != reference_names:
            raise ValueError(
                "A ordem ou quantidade de meta-features mudou entre datasets. "
                "Verifique se todos usam a mesma versão do pymfe."
            )

    meta_X = pd.DataFrame(
        [item["feature_values"] for item in meta_feature_results],
        columns=reference_names,
        index=[item["dataset_name"] for item in meta_feature_results],
    )
    save_dataframe(meta_X, META_X_PATH)
    print(f"meta_X salvo em: {META_X_PATH}")

print(f"Shape de meta_X: {meta_X.shape}")
meta_X.head()


## 9. Pré-processamento do meta-dataset e suporte ao GA

Contém: o pré-processador com filtro de correlação, o caching de fitness,
a penalidade dinâmica, e a execução do Algoritmo Genético.


In [ ]:
# ── Estrutura do pré-processador ──────────────────────────────────────────

@dataclass
class MetaPreprocessor:
    """
    Encapsula os transformadores ajustados no treino para aplicar no teste.

    Garante ausência de data leakage: todos os parâmetros (mediana de
    imputação, limiar de variância, correlações, média/desvio de escala)
    são aprendidos APENAS no conjunto de treino de cada fold externo e
    depois aplicados ao teste sem nenhuma informação adicional.

    Campos:
      imputer:              SimpleImputer ajustado no treino
      variance_filter:      VarianceThreshold ajustado no treino
      scaler:               StandardScaler ajustado no treino
      correlation_mask_:    booleanos — True = manter, False = descartada por correlação
      valid_input_columns_: colunas não-100%-nulas encontradas no treino
      selected_columns_:    colunas sobreviventes após variância + correlação (nomes)
    """
    imputer:              SimpleImputer
    variance_filter:      VarianceThreshold
    scaler:               StandardScaler
    correlation_mask_:    np.ndarray
    valid_input_columns_: np.ndarray
    selected_columns_:    np.ndarray

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        """Aplica sequencialmente todas as transformações aprendidas no treino."""
        X_work = X.replace([np.inf, -np.inf], np.nan)
        X_work = X_work.loc[:, self.valid_input_columns_]
        X_imp  = self.imputer.transform(X_work)
        X_filt = self.variance_filter.transform(X_imp)
        X_decorr = X_filt[:, self.correlation_mask_]
        return self.scaler.transform(X_decorr)


def remove_correlated_features(X: np.ndarray, threshold: float) -> np.ndarray:
    """
    Remove uma de cada par de meta-features com alta correlação de Pearson.

    PROBLEMA que esta função resolve:
      O meta-dataset do pymfe contém muitos pares com |r| > 0.90, como
      'joint_ent.mean' e 'joint_ent.sd', que carregam informação quase idêntica.
      Manter ambas infla o espaço de busca do GA: ele desperdica gerações
      tentando escolher entre features equivalentes, sem ganho real de fitness.

    ALGORITMO (greedy forward):
      Percorre features em ordem de índice. Para cada feature ainda marcada
      como 'keep', verifica se ela tem |r| > threshold com qualquer feature
      já aceita. Se sim, descarta-a (mantém a de índice menor do par).

    Retorna máscara booleana: True = manter, False = descartar.
    A redução típica é de ~148 → ~90–100 features (≈35% de compressão adicional).
    """
    corr = np.corrcoef(X, rowvar=False)  # matriz (n_features × n_features)
    n    = corr.shape[0]
    keep = np.ones(n, dtype=bool)

    for i in range(n):
        if not keep[i]:
            continue
        for j in range(i + 1, n):
            if keep[j] and abs(corr[i, j]) > threshold:
                keep[j] = False  # descarta j; mantém i

    return keep


def fit_meta_preprocessor(X_train: pd.DataFrame) -> MetaPreprocessor:
    """
    Ajusta o pipeline completo de pré-processamento do meta-dataset no treino.

    Pipeline de 5 etapas (todas ajustadas APENAS no conjunto de treino):

    Etapa 1 — Remoção de colunas 100% ausentes:
      Alguns meta-features do pymfe podem ser NaN em todos os datasets de
      um fold específico (ex: meta-features de landmarking em datasets muito
      simples). Removê-las evita desalinhamento entre treino e teste.

    Etapa 2 — Imputação pela mediana (SimpleImputer):
      Preenche NaNs restantes. A mediana é robusta a outliers e adequada
      para meta-features que podem ter distribuições assimétricas.

    Etapa 3 — Filtro de variância (VarianceThreshold):
      Remove meta-features com variância < 1e-8 (quasi-constantes).
      Features constantes não discriminam classes e só adicionam ruído.

    Etapa 4 — Filtro de correlação (remove_correlated_features) [NOVO]:
      Remove uma de cada par com |r| > 0.90. Reduz o espaço de busca do GA
      de ~148 para ~90-100 features, permitindo que o GA explore o espaço
      mais eficientemente com o mesmo orçamento de gerações.

    Etapa 5 — Padronização (StandardScaler):
      Centraliza e escala features para média 0 e desvio 1. A Random Forest
      (meta-modelo) é invariante à escala, mas features padronizadas facilitam
      a interpretação do fitness e a comparação de importâncias.
    """
    X_work = X_train.replace([np.inf, -np.inf], np.nan)

    # Etapa 1
    valid_cols = X_work.columns[~X_work.isna().all(axis=0)].to_numpy()
    X_work     = X_work.loc[:, valid_cols]

    # Etapa 2
    imputer   = SimpleImputer(strategy="median")
    X_imp     = imputer.fit_transform(X_work)

    # Etapa 3
    var_filt   = VarianceThreshold(threshold=1e-8)
    X_filt     = var_filt.fit_transform(X_imp)
    after_var  = valid_cols[var_filt.get_support()]

    # Etapa 4
    corr_mask  = remove_correlated_features(X_filt, CORRELATION_THRESHOLD)
    X_decorr   = X_filt[:, corr_mask]
    sel_cols   = after_var[corr_mask]
    print(f"  → Após variância: {X_filt.shape[1]} | "
          f"removidas por correlação: {(~corr_mask).sum()} | "
          f"restantes para o GA: {X_decorr.shape[1]}")

    # Etapa 5
    scaler = StandardScaler()
    scaler.fit(X_decorr)

    return MetaPreprocessor(
        imputer=imputer,
        variance_filter=var_filt,
        scaler=scaler,
        correlation_mask_=corr_mask,
        valid_input_columns_=valid_cols,
        selected_columns_=sel_cols,
    )


# ── Utilitários do GA ──────────────────────────────────────────────────────

def ensure_non_empty_mask(mask: np.ndarray, fallback_index: int = 0) -> np.ndarray:
    """
    Garante pelo menos uma feature ativa na máscara.

    O GA pode gerar soluções all-zero (nenhuma feature selecionada) em casos
    extremos de mutação agressiva. Isso causaria erro no treinamento do
    meta-modelo. Se isso ocorrer, ativa o gene de fallback_index como proteção.
    """
    mask = mask.astype(bool)
    if mask.sum() == 0:
        mask[fallback_index] = True
    return mask


def make_ga_fitness(
    X_train: np.ndarray,
    y_train: np.ndarray,
    alpha: float,
    threshold: float,
) -> callable:
    """
    Cria e retorna a função de fitness do GA com caching e penalidade dinâmica.

    ═══════════════════════════════════════════════════════════════════════
    CACHING DE FITNESS
    ═══════════════════════════════════════════════════════════════════════
    O GA opera com populações de soluções binárias. Entre gerações, a mesma
    máscara aparece repetidamente por três razões principais:

      1. Elitismo (keep_elitism=2): os 2 melhores são copiados intactos
         → fitness calculado novamente sem necessidade
      2. Convergência: nas gerações tardias a população homogeniza
         → muitos indivíduos idênticos surgem por crossover entre similares
      3. Mutação neutra: mutação com probabilidade baixa (0.05) pode reverter
         para o gene original → solução já vista reaparece

    Sem cache: até 50 gerações × 30 indivíduos = 1.500 chamadas de
    cross_val_score por fold. Com cache, repetições (30–50% das gerações
    tardias) retornam instantaneamente.

    Implementação: dict Python com chave = mask.tobytes() (hash exato bit a
    bit da máscara binária). Garante que só reutilizamos fitness de soluções
    IDÊNTICAS — sem falsos positivos por colisão de hash.

    ═══════════════════════════════════════════════════════════════════════
    MÉTRICA: F1-MACRO (era acurácia)
    ═══════════════════════════════════════════════════════════════════════
    Acurácia pura no meta-dataset desbalanceado é dominada pelas classes
    majoritárias (HistGradientBoosting + ExtraTrees ≈ 72%). O GA convergia
    para features que discriminam bem só essas classes, ignorando as demais.

    F1-macro calcula o F1 de cada classe separadamente e tira a média,
    dando peso igual a todas as classes independentemente da frequência.
    Isso força o GA a buscar features que também discriminem KNN, GaussianNB
    e DecisionTree — criando um gradiente de fitness mais rico e informativo.

    NOTA: a avaliação FINAL (compare com baseline, random, importance) usa
    acurácia para comparabilidade com a literatura de meta-learning.

    ═══════════════════════════════════════════════════════════════════════
    PENALIDADE DINÂMICA (era fixa = 0.03)
    ═══════════════════════════════════════════════════════════════════════
    Penalidade fixa tratava igualmente soluções com 20% e 70% das features.
    O GA não tinha incentivo para ser compacto além de um limiar mínimo.

    Penalidade dinâmica em dois regimes:
      • frac ≤ threshold (40%): penalty = alpha × frac
        → penalidade LINEAR suave: recompensa compactação moderada
      • frac > threshold:        penalty = alpha×thresh + alpha×2×(frac-thresh)²
        → penalidade QUADRÁTICA: pune fortemente soluções acima de 40%

    Isso cria um gradiente claro de incentivo para compacidade sem eliminar
    soluções levemente acima do threshold que podem ter melhor F1-macro.
    """
    n_features    = X_train.shape[1]
    fitness_cache: Dict[bytes, float] = {}  # cache local deste fold

    def fitness_func(ga_instance, solution, solution_idx):
        mask     = ensure_non_empty_mask(np.array(solution, dtype=int))
        mask_key = mask.tobytes()  # chave de cache: bytes exatos da máscara

        if mask_key in fitness_cache:
            return fitness_cache[mask_key]  # retorna do cache sem recalcular

        # Avalia qualidade da seleção via cross-validation com F1-macro.
        X_sel  = X_train[:, mask]
        scores = cross_val_score(
            clone(META_MODEL), X_sel, y_train,
            cv=INNER_META_CV, scoring="f1_macro",
            n_jobs=1,
        )
        f1 = float(np.mean(scores))

        # Penalidade dinâmica escalonada pela fração de features selecionadas.
        frac = mask.sum() / n_features
        if frac <= threshold:
            penalty = alpha * frac
        else:
            penalty = alpha * threshold + alpha * 2 * (frac - threshold) ** 2

        fitness = f1 - penalty
        fitness_cache[mask_key] = fitness  # armazena no cache
        return fitness

    return fitness_func


def run_ga_feature_selection(
    X_train: np.ndarray,
    y_train: np.ndarray,
    fold_seed: int,
) -> Tuple[np.ndarray, float, List[float]]:
    """
    Executa o Algoritmo Genético para seleção de meta-features neste fold.

    O GA evolui uma população de máscaras binárias, maximizando a função
    de fitness (F1-macro com penalidade dinâmica de dimensionalidade).

    ── Inicialização esparsa (35% ativas) ───────────────────────────────────
    A versão anterior usava ~50% de features ativas na população inicial
    (numpy.random.integers uniforme). Isso posicionava o GA na região central
    do espaço de busca — longe do ótimo esperado (20–45% de features).

    Agora usamos inicialização esparsa: cada gene é 1 com probabilidade 0.35.
    A população começa mais próxima das soluções compactas que a penalidade
    dinâmica favorece, acelerando a convergência para o trade-off ideal.

    ── Histórico de convergência ────────────────────────────────────────────
    O callback on_generation registra o melhor fitness de CADA geração.
    Na versão anterior só o fold 1 era salvo — agora salvamos todos os folds,
    permitindo diagnóstico completo de convergência por fold.

    Retorna:
      best_mask:    máscara booleana das features selecionadas
      best_fitness: fitness da melhor solução encontrada
      history:      lista do melhor fitness por geração (para curvas de convergência)
    """
    n_features = X_train.shape[1]
    rng        = np.random.default_rng(fold_seed)

    # Inicialização esparsa: ~35% de features ativas (era ~50% aleatório).
    initial_population = (
        rng.random(size=(GA_CONFIG["sol_per_pop"], n_features)) < 0.35
    ).astype(int)

    history = []

    def on_generation(ga_instance):
        """Registra o melhor fitness desta geração no histórico de convergência."""
        best = float(ga_instance.best_solution(
            pop_fitness=ga_instance.last_generation_fitness
        )[1])
        history.append(best)

    ga_instance = pygad.GA(
        num_genes=n_features,
        fitness_func=make_ga_fitness(
            X_train, y_train, GA_PENALTY_ALPHA, GA_PENALTY_THRESHOLD,
        ),
        initial_population=initial_population,
        random_seed=fold_seed,
        on_generation=on_generation,
        **GA_CONFIG,
    )
    ga_instance.run()

    best_solution, best_fitness, _ = ga_instance.best_solution()
    best_mask = ensure_non_empty_mask(np.array(best_solution, dtype=int))
    return best_mask, float(best_fitness), history


# ── Funções de avaliação e baselines ──────────────────────────────────────

def evaluate_meta_strategy(
    X_train: np.ndarray, y_train: np.ndarray,
    X_test:  np.ndarray, y_test:  np.ndarray,
    mask: np.ndarray,
) -> float:
    """
    Treina o meta-modelo com a seleção dada e avalia a acurácia no teste.

    Usamos acurácia (não F1) na avaliação final para comparabilidade
    com a literatura de meta-learning. O F1-macro é usado apenas
    internamente no fitness do GA para guiar a busca evolutiva.
    """
    mask  = ensure_non_empty_mask(mask)
    model = clone(META_MODEL)
    model.fit(X_train[:, mask], y_train)
    return float(accuracy_score(y_test, model.predict(X_test[:, mask])))


def get_topk_importance_mask(
    X_train: np.ndarray, y_train: np.ndarray, k: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Seleciona as top-k features por importância de Gini da Random Forest.

    Treina o meta-modelo completo e ordena as features pelo atributo
    feature_importances_. Retorna a máscara das k mais importantes.
    Usado como baseline determinístico (sem aleatoriedade) para comparação.
    """
    model = clone(META_MODEL)
    model.fit(X_train, y_train)
    importances = model.feature_importances_
    topk_idx    = np.argsort(importances)[::-1][:k]
    mask        = np.zeros(X_train.shape[1], dtype=bool)
    mask[topk_idx] = True
    return mask, importances


def get_random_mask(n_features: int, k: int, seed: int) -> np.ndarray:
    """
    Seleciona k features aleatoriamente como baseline estocástico.

    Usa o mesmo orçamento k escolhido pelo GA para comparação justa.
    Seeds diferentes por fold garantem independência entre as amostras.
    Representa o 'piso' esperado: qualquer método razoável deve superá-lo.
    """
    rng    = np.random.default_rng(seed)
    chosen = rng.choice(n_features, size=max(1, k), replace=False)
    mask   = np.zeros(n_features, dtype=bool)
    mask[chosen] = True
    return mask


## 10. Fase 3–5 — Validação externa e avaliação comparativa

O loop de StratifiedKFold 5-fold simula generalização para datasets novos.
Em cada fold: pré-processa, roda o GA, avalia 4 estratégias no teste.


In [ ]:
if (OUTER_RESULTS_PATH.exists()
        and TOP_FEATURES_COUNTER_PATH.exists()
        and ALL_FOLDS_HISTORY_PATH.exists()):
    results_df               = load_dataframe(OUTER_RESULTS_PATH)
    selected_feature_counter = load_pickle(TOP_FEATURES_COUNTER_PATH)
    all_folds_ga_history     = load_pickle(ALL_FOLDS_HISTORY_PATH)
    first_fold_ga_history    = all_folds_ga_history[0]
    print(f"Resultados carregados do cache: {OUTER_RESULTS_PATH}")

else:
    print("Executando avaliação comparativa com GA (pode demorar)...\n")
    outer_fold_results       = []
    selected_feature_counter = Counter()
    all_folds_ga_history     = []

    for fold_idx, (train_idx, test_idx) in enumerate(
        OUTER_CV.split(meta_X, meta_y), start=1
    ):
        print(f"{'='*60}")
        print(f"Fold externo {fold_idx}/5")
        print(f"{'='*60}")

        # Divide o meta-dataset em treino e teste para este fold.
        # StratifiedKFold garante que todas as classes apareçam no treino,
        # evitando folds onde classes minoritárias desaparecem completamente.
        X_train_df = meta_X.iloc[train_idx].copy()
        X_test_df  = meta_X.iloc[test_idx].copy()
        y_train    = meta_y[train_idx]
        y_test     = meta_y[test_idx]
        print(f"Treino: {len(train_idx)} datasets | Teste: {len(test_idx)} datasets")

        # Pré-processamento ajustado APENAS no treino (sem data leakage).
        # Inclui: remoção de NaN totais, imputação, variância,
        # filtro de correlação [NOVO] e padronização.
        preprocessor = fit_meta_preprocessor(X_train_df)
        X_train      = preprocessor.transform(X_train_df)
        X_test       = preprocessor.transform(X_test_df)
        surviving_names = preprocessor.selected_columns_
        print(f"Meta-features para o GA: {X_train.shape[1]}")

        # Baseline: usa todas as features após pré-processamento (100%).
        # É a referência superior — a melhor que podemos esperar sem seleção.
        baseline_mask = np.ones(X_train.shape[1], dtype=bool)

        # GA: busca evolutiva com caching de fitness e penalidade dinâmica.
        print(f"Rodando GA (seed={RANDOM_STATE + fold_idx})...")
        ga_mask, ga_fitness, ga_history = run_ga_feature_selection(
            X_train, y_train, fold_seed=RANDOM_STATE + fold_idx,
        )
        all_folds_ga_history.append(ga_history)
        k_selected = int(ga_mask.sum())
        print(f"GA selecionou {k_selected}/{X_train.shape[1]} features | "
              f"fitness={ga_fitness:.4f}")

        # Baselines com o mesmo orçamento k para comparação justa.
        random_mask, _     = get_random_mask(
            X_train.shape[1], k_selected,
            seed=RANDOM_STATE + 1000 + fold_idx,
        ), None
        random_mask        = get_random_mask(
            X_train.shape[1], k_selected,
            seed=RANDOM_STATE + 1000 + fold_idx,
        )
        importance_mask, _ = get_topk_importance_mask(X_train, y_train, k_selected)

        # Avaliação das 4 estratégias no conjunto de TESTE.
        baseline_acc   = evaluate_meta_strategy(X_train, y_train, X_test, y_test, baseline_mask)
        ga_acc         = evaluate_meta_strategy(X_train, y_train, X_test, y_test, ga_mask)
        random_acc     = evaluate_meta_strategy(X_train, y_train, X_test, y_test, random_mask)
        importance_acc = evaluate_meta_strategy(X_train, y_train, X_test, y_test, importance_mask)
        print(f"Baseline={baseline_acc:.4f} | GA={ga_acc:.4f} | "
              f"Random={random_acc:.4f} | Importance={importance_acc:.4f}")

        # Contabiliza quais meta-features o GA incluiu neste fold.
        selected_feature_counter.update(surviving_names[ga_mask])

        outer_fold_results.append({
            "fold":                fold_idx,
            "n_features_total":    int(X_train.shape[1]),
            "n_features_ga":       k_selected,
            "reduction_pct":       float(100 * (1 - k_selected / X_train.shape[1])),
            "ga_fitness":          ga_fitness,
            "baseline_accuracy":   baseline_acc,
            "ga_accuracy":         ga_acc,
            "random_accuracy":     random_acc,
            "importance_accuracy": importance_acc,
        })

    results_df = pd.DataFrame(outer_fold_results)
    first_fold_ga_history = all_folds_ga_history[0]

    save_dataframe(results_df, OUTER_RESULTS_PATH)
    save_pickle(selected_feature_counter, TOP_FEATURES_COUNTER_PATH)
    save_pickle(all_folds_ga_history, ALL_FOLDS_HISTORY_PATH)
    print(f"\nResultados salvos em: {OUTER_RESULTS_PATH}")

results_df


## 11. Tabela consolidada de desempenho

In [ ]:
# Consolida média e desvio padrão das 4 estratégias nos 5 folds externos.
# Desvio padrão alto = estratégia instável entre partições do meta-dataset.
summary_table = pd.DataFrame({
    "Abordagem": ["Baseline", "GA", "Random Feature Selection", "Feature Importance"],
    "Acurácia média no meta-teste": [
        results_df["baseline_accuracy"].mean(),
        results_df["ga_accuracy"].mean(),
        results_df["random_accuracy"].mean(),
        results_df["importance_accuracy"].mean(),
    ],
    "Desvio padrão": [
        results_df["baseline_accuracy"].std(),
        results_df["ga_accuracy"].std(),
        results_df["random_accuracy"].std(),
        results_df["importance_accuracy"].std(),
    ],
}).sort_values("Acurácia média no meta-teste", ascending=False).reset_index(drop=True)

summary_table.style.format({
    "Acurácia média no meta-teste": "{:.4f}",
    "Desvio padrão": "{:.4f}",
})


## 12. Redução média de dimensionalidade

In [ ]:
avg_total = results_df["n_features_total"].mean()
avg_ga    = results_df["n_features_ga"].mean()
avg_red   = results_df["reduction_pct"].mean()
print(f"Meta-features após pré-processamento: {avg_total:.2f}")
print(f"Meta-features selecionadas pelo GA:   {avg_ga:.2f}")
print(f"Redução percentual média:             {avg_red:.2f}%")


## 13. Curvas de convergência do GA — todos os folds

Agora exibimos a convergência de TODOS os folds (antes só o fold 1 era salvo).
Uma curva ascendente e depois estável indica que o GA convergiu adequadamente.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)
for fold_idx, (history, ax) in enumerate(zip(all_folds_ga_history, axes), start=1):
    ax.plot(np.arange(1, len(history) + 1), history, marker="o", linewidth=2, markersize=4)
    ax.set_title(f"Fold {fold_idx}")
    ax.set_xlabel("Geração")
    if fold_idx == 1:
        ax.set_ylabel("Fitness (F1-macro - penalidade)")
    ax.grid(True, alpha=0.3)
fig.suptitle("Curvas de convergência do GA por fold externo", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## 14. Meta-features mais recorrentes nas soluções do GA

In [ ]:
top_features_df = pd.DataFrame(
    selected_feature_counter.most_common(10),
    columns=["meta_feature", "frequencia"],
)
plt.figure(figsize=(12, 6))
sns.barplot(data=top_features_df, x="frequencia", y="meta_feature", palette="viridis")
plt.title("Top 10 meta-features mais selecionadas pelo GA")
plt.xlabel("Frequência de seleção (máx = 5 folds)")
plt.ylabel("Meta-feature")
plt.tight_layout()
plt.show()
top_features_df


## 15. Exportação das tabelas para Excel

In [ ]:
import openpyxl

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

catalog_df.to_excel(INTERMEDIATE_TABLES_DIR / "01_openml_catalog.xlsx", index=False)
base_results_df.to_excel(INTERMEDIATE_TABLES_DIR / "02_base_level_results.xlsx", index=False)
meta_X.to_excel(INTERMEDIATE_TABLES_DIR / "03_meta_features_matrix.xlsx", index=True)
results_df.to_excel(INTERMEDIATE_TABLES_DIR / "04_outer_cv_results.xlsx", index=False)
summary_table.to_excel(INTERMEDIATE_TABLES_DIR / "05_summary_table.xlsx", index=False)
top_features_df.to_excel(INTERMEDIATE_TABLES_DIR / "06_top_ga_features.xlsx", index=False)
print(f"Tabelas exportadas para: {INTERMEDIATE_TABLES_DIR}")


## 16. Gráficos adicionais de análise

In [ ]:
# ── Prepara estruturas auxiliares ─────────────────────────────────────────
meta_label_counts = pd.Series(meta_y).value_counts().reset_index()
meta_label_counts.columns = ["meta_label", "count"]

perf_long = results_df.melt(
    id_vars=["fold"],
    value_vars=["baseline_accuracy","ga_accuracy","random_accuracy","importance_accuracy"],
    var_name="approach", value_name="accuracy",
)
perf_long["approach"] = perf_long["approach"].map({
    "baseline_accuracy": "Baseline", "ga_accuracy": "GA",
    "random_accuracy": "Random", "importance_accuracy": "Feature Importance",
})

results_corr = results_df[[
    "baseline_accuracy","ga_accuracy","random_accuracy","importance_accuracy"
]].corr()

# Distribuição das meta-labels
plt.figure(figsize=(10, 5))
sns.barplot(data=meta_label_counts, x="meta_label", y="count", palette="crest")
plt.title("Distribuição das meta-labels no meta-dataset")
plt.xlabel("Melhor algoritmo base por dataset")
plt.ylabel("Quantidade de datasets")
plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()

# Boxplot por abordagem
plt.figure(figsize=(10, 6))
sns.boxplot(data=perf_long, x="approach", y="accuracy", palette="Set2")
sns.stripplot(data=perf_long, x="approach", y="accuracy", color="black", alpha=0.6, size=5)
plt.title("Distribuição da acurácia por abordagem nos folds externos")
plt.xlabel("Abordagem"); plt.ylabel("Acurácia")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

# Redução por fold
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="fold", y="reduction_pct", palette="flare")
plt.title("Redução percentual de meta-features pelo GA por fold")
plt.xlabel("Fold externo"); plt.ylabel("Redução (%)")
plt.tight_layout(); plt.show()

# Correlação entre abordagens
plt.figure(figsize=(8, 6))
sns.heatmap(results_corr, annot=True, cmap="YlGnBu", fmt=".2f", square=True)
plt.title("Correlação entre acurácias das abordagens por fold")
plt.tight_layout(); plt.show()

# Features GA vs acurácia GA
plt.figure(figsize=(10, 5))
sns.scatterplot(data=results_df, x="n_features_ga", y="ga_accuracy",
                hue="fold", palette="tab10", s=120)
plt.title("Features selecionadas vs acurácia do GA")
plt.xlabel("Nº features selecionadas"); plt.ylabel("Acurácia GA")
plt.tight_layout(); plt.show()

# Ganho do GA
gain_df = results_df[["fold","ga_accuracy","baseline_accuracy","random_accuracy","importance_accuracy"]].copy()
gain_df["GA - Baseline"]           = gain_df["ga_accuracy"] - gain_df["baseline_accuracy"]
gain_df["GA - Random"]             = gain_df["ga_accuracy"] - gain_df["random_accuracy"]
gain_df["GA - Feature Importance"] = gain_df["ga_accuracy"] - gain_df["importance_accuracy"]
gain_long = gain_df[["fold","GA - Baseline","GA - Random","GA - Feature Importance"]].melt(
    id_vars="fold", var_name="comparison", value_name="gain"
)
plt.figure(figsize=(10, 5))
sns.barplot(data=gain_long, x="comparison", y="gain", palette="coolwarm")
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Ganho médio do GA em relação às demais abordagens")
plt.xlabel("Comparação"); plt.ylabel("Diferença média de acurácia")
plt.xticks(rotation=10); plt.tight_layout(); plt.show()


## 17. Análise interpretativa dos resultados

### Análise 1 — Comparação geral entre as abordagens

Compara a acurácia média ± desvio padrão das 4 estratégias.
Se o GA superar ou empatar com o Baseline usando menos features,
há um bom trade-off acurácia/dimensionalidade.


In [ ]:
plt.figure(figsize=(10, 5))
plot_df = summary_table.copy()
sns.barplot(data=plot_df, x="Abordagem", y="Acurácia média no meta-teste", palette="Set2")
plt.errorbar(
    x=np.arange(len(plot_df)),
    y=plot_df["Acurácia média no meta-teste"],
    yerr=plot_df["Desvio padrão"],
    fmt="none", ecolor="black", capsize=5,
)
plt.title("Comparação geral das abordagens no meta-teste")
plt.xlabel("Abordagem"); plt.ylabel("Acurácia média ± desvio padrão")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()
display(summary_table)


### Análise 2 — Redução de dimensionalidade pelo GA

In [ ]:
avg_total   = results_df["n_features_total"].mean()
avg_ga      = results_df["n_features_ga"].mean()
avg_red     = results_df["reduction_pct"].mean()
avg_removed = avg_total - avg_ga
print(f"GA reduziu em média {avg_red:.2f}% das meta-features, "
      f"mantendo {avg_ga:.2f} de {avg_total:.2f} ({avg_removed:.2f} removidas por fold).")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(
    data=pd.DataFrame({
        "categoria": ["Baseline (todas)", "GA (mantidas)", "Removidas"],
        "quantidade": [avg_total, avg_ga, avg_removed],
    }),
    x="categoria", y="quantidade", palette="pastel", ax=axes[0],
)
axes[0].set_title("Média: Baseline vs GA")
axes[0].set_xlabel(""); axes[0].set_ylabel("Quantidade média")
axes[0].tick_params(axis="x", rotation=12)

comp_by_fold = pd.DataFrame({
    "fold":    results_df["fold"].tolist() * 2,
    "tipo":    ["Baseline"]*len(results_df) + ["GA"]*len(results_df),
    "n_feats": results_df["n_features_total"].tolist() + results_df["n_features_ga"].tolist(),
})
sns.barplot(data=comp_by_fold, x="fold", y="n_feats", hue="tipo",
            palette=["#DD8452","#4C72B0"], ax=axes[1])
axes[1].set_title("Baseline vs GA por fold")
axes[1].set_xlabel("Fold"); axes[1].set_ylabel("Nº de meta-features")
plt.tight_layout(); plt.show()
display(results_df[["fold","n_features_total","n_features_ga","reduction_pct"]])


### Análise 3 — Estabilidade do GA entre os folds

In [ ]:
comp_fold = results_df[["fold","baseline_accuracy","ga_accuracy",
                        "random_accuracy","importance_accuracy"]].melt(
    id_vars="fold", var_name="approach", value_name="accuracy"
)
comp_fold["approach"] = comp_fold["approach"].map({
    "baseline_accuracy":"Baseline","ga_accuracy":"GA",
    "random_accuracy":"Random","importance_accuracy":"Feature Importance",
})
plt.figure(figsize=(10, 5))
sns.lineplot(data=comp_fold, x="fold", y="accuracy", hue="approach", marker="o", linewidth=2.5)
plt.title("Desempenho das abordagens ao longo dos folds externos")
plt.xlabel("Fold"); plt.ylabel("Acurácia no meta-teste")
plt.tight_layout(); plt.show()
display(results_df[["fold","baseline_accuracy","ga_accuracy","random_accuracy","importance_accuracy"]])


### Análise 4 — Ganho do GA em relação às alternativas

In [ ]:
gain_df = results_df[["fold","ga_accuracy","baseline_accuracy",
                       "random_accuracy","importance_accuracy"]].copy()
gain_df["GA - Baseline"]           = gain_df["ga_accuracy"] - gain_df["baseline_accuracy"]
gain_df["GA - Random"]             = gain_df["ga_accuracy"] - gain_df["random_accuracy"]
gain_df["GA - Feature Importance"] = gain_df["ga_accuracy"] - gain_df["importance_accuracy"]
gain_long = gain_df[["fold","GA - Baseline","GA - Random","GA - Feature Importance"]].melt(
    id_vars="fold", var_name="comparison", value_name="gain"
)
plt.figure(figsize=(10, 5))
sns.barplot(data=gain_long, x="comparison", y="gain", palette="coolwarm")
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.title("Ganho médio do GA em relação às demais abordagens")
plt.xlabel("Comparação"); plt.ylabel("Diferença média de acurácia")
plt.xticks(rotation=10); plt.tight_layout(); plt.show()
display(gain_long.groupby("comparison", as_index=False)["gain"].mean())


### Análise 5 — Distribuição das meta-labels

In [ ]:
dist = pd.Series(meta_y).value_counts().reset_index()
dist.columns = ["meta_label", "count"]
plt.figure(figsize=(10, 5))
sns.barplot(data=dist, x="meta_label", y="count", palette="crest")
plt.title("Distribuição das meta-labels no meta-dataset")
plt.xlabel("Melhor algoritmo base"); plt.ylabel("Quantidade de datasets")
plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()
display(dist)


### Análise 6 — Meta-features preferidas pelo GA

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_features_df, x="frequencia", y="meta_feature", palette="viridis")
plt.title("Meta-features mais recorrentes nas soluções do GA")
plt.xlabel("Frequência de seleção (máx = 5 folds)"); plt.ylabel("Meta-feature")
plt.tight_layout(); plt.show()
display(top_features_df)
